In [ ]:
# Base Model & LoRA Adapter Merging Pipeline

**Objective:**
This notebook takes the trained LoRA adapters from the Supervised Fine-Tuning (SFT) phase and permanently fuses them with the base `Gemma-4-26b` model. 

**Why is this necessary?**
During training, we only updated a small fraction of the model's parameters (the adapters) to save compute costs. However, for high-speed production inference on our 24-hour Snowflake/AWS batch pipeline, we need a single, unified model. This script executes a `merge_and_unload()` operation to bake the categorical logic into the core model weights, outputting a brand-new, fully standalone AI model.

In [ ]:
# Install required dependencies for model merging
!pip install -q transformers peft torch accelerate huggingface_hub

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# --- Configuration & File Paths ---
# INSTRUCTION: Ensure ADAPTER_DIR points to the output of your 100% training run.
BASE_MODEL_ID = "google/gemma-4-26b"
ADAPTER_DIR = "./aws_weights/gemma4_26b_100_percent"

# Define the final output directory and the new model's official name
EXPORT_DIR = "./production_model"
NEW_MODEL_NAME = "Buyee-Gemma-4-26B-Categorization-V1"

FINAL_SAVE_PATH = os.path.join(EXPORT_DIR, NEW_MODEL_NAME)
os.makedirs(FINAL_SAVE_PATH, exist_ok=True)

# Security & API Setup
# INSTRUCTION: Insert your Hugging Face access token here
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"

print(f"Configuration set.")
print(f"Base Model: {BASE_MODEL_ID}")
print(f"Adapters loading from: {ADAPTER_DIR}")
print(f"🚀 New Model will be saved to: {FINAL_SAVE_PATH}")

In [ ]:
# --- Load the Base Model ---
print("Loading the base model into memory. (This requires high VRAM capacity)...")

# We load the model in bfloat16 (16-bit) rather than 4-bit. 
# You cannot mathematically merge weights into a dynamically quantized 4-bit model.
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

# Load the tokenizer (crucial to save alongside the new model)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)

print("✅ Base model and tokenizer successfully loaded.")

In [ ]:
# --- Merge Adapters and Export New Model ---
print(f"Fusing LoRA adapters from {ADAPTER_DIR} into the base model...")

# 1. Wrap the base model with the trained Peft adapters
peft_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Execute the fusion operation
# This mathematically combines the adapter weights into the base linear layers.
print("Baking weights... (This may take a few minutes)")
unified_model = peft_model.merge_and_unload()

print("✅ Fusion complete. Model is now a standalone entity.")

# 3. Save the newly minted model and tokenizer to local storage
print(f"💾 Saving '{NEW_MODEL_NAME}' to disk at: {FINAL_SAVE_PATH}")

unified_model.save_pretrained(FINAL_SAVE_PATH, safe_serialization=True)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

print("\n=======================================================")
print(f"🎉 SUCCESS! Your new model is ready for production.")
print(f"Model Name: {NEW_MODEL_NAME}")
print(f"Location: {FINAL_SAVE_PATH}")
print("You can now point your automated 24-hour inference scripts directly to this path.")
print("=======================================================")